# Burst Experiment — Simplified 3-Phase Pipeline

Three independent phases:
1. **Pretrain** — train on background tasks until convergence (run once)
2. **Finetune** — burst phase with configurable concentration (sweep here)
3. **Forget** — reversion phase, measure how quickly burst knowledge is lost

Each phase saves checkpoints so you can re-run downstream phases without retraining upstream.

In [4]:
import sys, os
os.chdir("/workspace/compositional_capabilities_burst")
sys.path.insert(0, ".")
from burst.simple import make_data, pretrain, finetune, forget, report
import matplotlib.pyplot as plt
%matplotlib inline

OUT = "data/simple_burst"

ModuleNotFoundError: No module named 'numpy'

## 1. Generate Data

One call, reuse across all phases. Adjust `depth`, `burst_pos`, `n_a` to change task complexity.

In [ ]:
data = make_data(depth=3, burst_pos=3, n_a=4)
print(f"Vocab: {data['vocab_size']}, Context: {data['context_size']}, Prompt: {data['prompt_len']}")
print(f"Other tasks: {data['task_info']['n_other_tasks']}, Burst tasks: {data['task_info']['n_burst_tasks']}")

## 2. Pretrain (run once)

Trains on background data until acc_other >= 0.99. Saves checkpoint to reuse.

In [ ]:
pt = pretrain(data, f"{OUT}/pretrain", steps=600)
report.plot_pretrain(pt)
plt.show()
print(f"Checkpoint: {pt['ckpt_path']}")

## 3. Finetune — sweep over burst fractions

Each run loads the same pretrained checkpoint. Edit `FRACS` to test different concentrations.

In [ ]:
FRACS = [1.0, 0.5, 0.25, 0.10]  # sweep these

ft_results = []
for frac in FRACS:
    r = finetune(data, pt["ckpt_path"], f"{OUT}/finetune",
                 burst_frac=frac, steps=200)
    ft_results.append(r)
    print(f"  {r['tag']}: peak_burst={r['peak_burst']:.3f}")

report.plot_finetune(ft_results)
plt.show()

## 4. Forget — measure reversion for each finetuned model

In [ ]:
fg_results = []
for ft in ft_results:
    r = forget(data, ft["ckpt_path"], f"{OUT}/forget", steps=300)
    fg_results.append(r)
    print(f"  {r['tag']}: peak={r['peak_burst']:.3f} -> end={r['end_burst_acc']:.3f} "
          f"(drop {r['dropoff_pct']:.1f}%)")

report.plot_forget(fg_results)
plt.show()

## 5. Report

In [ ]:
# Summary table
rows = report.summary_table(ft_results, fg_results)

In [ ]:
# Full trajectory + comparison charts
fig = report.plot_full_trajectory(pt, ft_results, fg_results)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
report.plot_peak_vs_frac(ft_results, axes[0])
report.plot_retention_vs_frac(fg_results, axes[1])
fig.tight_layout()
plt.show()

In [ ]:
# Save all charts to disk
report.save_report(pt, ft_results, fg_results, f"{OUT}/report")